# Notebook 03 — Long-range social pressure, uncorrelated duplex

**Goal:** Study the effect of vigilance range L ∈ {1, 2, 3, 4} when the game layer and the
vigilance layer are **independent realizations** of the same random network model
(uncorrelated duplex). Comparison with notebook 02 isolates the role of inter-layer correlation.

Networks: BA and ER, z = 4 and z = 16.  
Update rule: Fermi (K = 0.1).  
Parameters: b ∈ [1.0, 2.0], θ ∈ [0.0, 1.0], N = 1000, 100 replications.

**Structure:**
- **Part I — Simulation:** runs sweeps, saves to `data/03-*.csv`. Run once.
- **Part II — Figures:** loads CSVs, makes all plots independently.
  Includes comparison figures against notebook 02 (correlated).

In [ ]:
# ── Shared: imports and parameters ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import model

N        = 1000
n_rep    = 100
L_VALUES = [1, 2, 3, 4]
lam      = 0.5
K_fermi  = 0.1
b_vals   = np.round(np.linspace(1.0, 2.0, 11), 2)
th_vals  = np.round(np.linspace(0.0, 1.0, 11), 2)
NET_SEED     = 0      # seed for game layer network
NET_VIG_SEED = 100    # different seed → independent vigilance layer network
SIM_SEED     = 0
N_JOBS       = -1

NETWORKS = [
    ('BA_z4',  'BA', 4,  r'BA, $z=4$'),
    ('BA_z16', 'BA', 16, r'BA, $z=16$'),
    ('ER_z4',  'ER', 4,  r'ER, $z=4$'),
    ('ER_z16', 'ER', 16, r'ER, $z=16$'),
]

os.makedirs('data',    exist_ok=True)
os.makedirs('figures', exist_ok=True)
print('b:', b_vals)
print('θ:', th_vals)

---
## Part I — Simulation
*Run once. G_game and G_vig are independent realizations (different random seeds).*

In [ ]:
# ── Build network pairs ──────────────────────────────────────────────────────
# G_game: used for payoffs and strategy update.
# G_vig:  used for vigilance influence (independent realization, same parameters).
game_graphs = {
    key: model.build_network(topo, N, z, seed=NET_SEED)
    for key, topo, z, _ in NETWORKS
}
vig_graphs = {
    key: model.build_network(topo, N, z, seed=NET_VIG_SEED)
    for key, topo, z, _ in NETWORKS
}

for key, topo, z, label in NETWORKS:
    Gg = game_graphs[key]
    Gv = vig_graphs[key]
    print(
        f"{label:14s}  "
        f"game: <k>={2*Gg.number_of_edges()/Gg.number_of_nodes():.2f} diam={nx.diameter(Gg)}  "
        f"vig:  <k>={2*Gv.number_of_edges()/Gv.number_of_nodes():.2f} diam={nx.diameter(Gv)}"
    )

In [ ]:
# ── Pre-compile JIT kernels ─────────────────────────────────────────────────
model.warm_up()
print('JIT kernels compiled.')

In [ ]:
# ── Run sweeps (uncorrelated: G_vig ≠ G_game) ───────────────────────────────
for key, topo, z, label in NETWORKS:
    Gg     = game_graphs[key]
    Gv     = vig_graphs[key]
    gp, gd = model.game_csr(Gg)   # game layer CSR
    for L in L_VALUES:
        path = f"data/03-{key}-L{L}.csv"
        if os.path.exists(path):
            print(f"  skip: {path}")
            continue
        print(f"  running {label}  L={L} ...", flush=True)
        sp, sd = model.shells_csr(Gv, L)   # vigilance shells from G_vig
        al     = model.geometric_kernel(L, lam)
        df = model.run_sweep(
            gp, gd, sp, sd, al,
            b_vals, th_vals,
            K=K_fermi, n_rep=n_rep,
            n_jobs=N_JOBS, base_seed=SIM_SEED,
            update_rule='fermi',
        )
        df.to_csv(path, index=False)
        print(f"  saved: {path}")

print('Part I done.')

---
## Part II — Figures
*Loads CSVs from `data/`. For correlated vs uncorrelated comparison, also loads `data/02-*.csv`.*

In [ ]:
# ── Load results ─────────────────────────────────────────────────────────────
data_unc = {}   # uncorrelated (this notebook)
data_cor = {}   # correlated   (notebook 02, for comparison)

for key, topo, z, label in NETWORKS:
    for L in L_VALUES:
        data_unc[(key, L)] = pd.read_csv(f"data/03-{key}-L{L}.csv")
        data_cor[(key, L)] = pd.read_csv(f"data/02-{key}-L{L}.csv")
        print(f"  loaded 03 and 02  {key} L={L}")

In [ ]:
# ── Figure settings ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 120,
})

CMAP       = 'Blues'
CMAP_DIFF  = 'RdBu'
VMIN, VMAX = 0.0, 1.0
VDIFF      = 0.4
THETA_SHOW = [0.3, 0.5, 0.7]
L_COLORS   = {1: 'C0', 2: 'C1', 3: 'C2', 4: 'C3'}

db, dt = b_vals[1] - b_vals[0], th_vals[1] - th_vals[0]
EXTENT = [b_vals[0]-db/2, b_vals[-1]+db/2, th_vals[0]-dt/2, th_vals[-1]+dt/2]

def rho_matrix(df):
    return (
        df.pivot(index='theta', columns='b', values='rho_mean')
          .sort_index(ascending=True)
          .values
    )

In [ ]:
# ── Figure 1: heatmaps — uncorrelated duplex, rows = networks, cols = L ──────
fig1, axes = plt.subplots(
    len(NETWORKS), len(L_VALUES),
    figsize=(11, 12),
    constrained_layout=True,
)

for row, (key, topo, z, label) in enumerate(NETWORKS):
    for col, L in enumerate(L_VALUES):
        ax  = axes[row, col]
        mat = rho_matrix(data_unc[(key, L)])
        im  = ax.imshow(
            mat, origin='lower', aspect='auto',
            vmin=VMIN, vmax=VMAX, cmap=CMAP, extent=EXTENT,
        )
        ax.set_xlabel('$b$')
        ax.set_ylabel('$\\theta$')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        if row == 0:
            ax.set_title(f'$L={L}$', fontweight='bold')
        if col == 0:
            ax.text(
                -0.45, 0.5, label, transform=ax.transAxes,
                rotation=90, va='center', ha='center', fontsize=9,
            )
        cb = fig1.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.set_label('$\\langle\\rho\\rangle$')
        cb.set_ticks([0, 0.5, 1.0])

fig1.suptitle(
    'Uncorrelated duplex — Fermi, $\\lambda=0.5$', y=1.01,
)
fig1.savefig('figures/03-heatmaps.pdf', bbox_inches='tight')
fig1.savefig('figures/03-heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03-heatmaps.{pdf,png}')

In [ ]:
# ── Figure 2: correlated vs uncorrelated — difference maps ────────────────────
# Δρ = rho_uncorrelated - rho_correlated.
# Blue = uncorrelated promotes more cooperation; red = correlated is better.
L_SHOW = L_VALUES   # show all L values

fig2, axes2 = plt.subplots(
    len(NETWORKS), len(L_SHOW),
    figsize=(11, 12),
    constrained_layout=True,
)

for row, (key, topo, z, label) in enumerate(NETWORKS):
    for col, L in enumerate(L_SHOW):
        ax   = axes2[row, col]
        diff = rho_matrix(data_unc[(key, L)]) - rho_matrix(data_cor[(key, L)])
        im   = ax.imshow(
            diff, origin='lower', aspect='auto',
            vmin=-VDIFF, vmax=VDIFF, cmap=CMAP_DIFF, extent=EXTENT,
        )
        ax.set_xlabel('$b$')
        ax.set_ylabel('$\\theta$')
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        if row == 0:
            ax.set_title(f'$L={L}$', fontweight='bold')
        if col == 0:
            ax.text(
                -0.45, 0.5, label, transform=ax.transAxes,
                rotation=90, va='center', ha='center', fontsize=9,
            )
        cb = fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.set_label('$\\Delta\\langle\\rho\\rangle$')
        cb.set_ticks([-VDIFF, 0, VDIFF])

fig2.suptitle(
    '$\\langle\\rho\\rangle_{\\rm uncorr} - \\langle\\rho\\rangle_{\\rm corr}$ — role of inter-layer correlation',
    y=1.01,
)
fig2.savefig('figures/03-corr-vs-uncorr.pdf', bbox_inches='tight')
fig2.savefig('figures/03-corr-vs-uncorr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03-corr-vs-uncorr.{pdf,png}')

In [ ]:
# ── Figure 3: marginals rho vs b — correlated vs uncorrelated at each L ───────
# One row per θ value, one column per network.
# Lines: solid = correlated, dashed = uncorrelated, colour = L.
TH_PLOT = 0.5   # single theta for clarity

fig3, axes3 = plt.subplots(
    1, len(NETWORKS),
    figsize=(11, 3.5),
    constrained_layout=True,
    sharey=True,
)

for col, (key, topo, z, label) in enumerate(NETWORKS):
    ax = axes3[col]
    for L in L_VALUES:
        for src, ls, data_dict in [
            ('corr',  '-',  data_cor),
            ('uncorr','--', data_unc),
        ]:
            df  = data_dict[(key, L)]
            sub = df[np.isclose(df['theta'], TH_PLOT)].sort_values('b')
            lbl = f'$L={L}$ {src}' if col == 0 else None
            ax.plot(
                sub['b'], sub['rho_mean'],
                ls, color=L_COLORS[L], lw=1.5, label=lbl,
            )
    ax.set_title(label)
    ax.set_xlabel('$b$')
    if col == 0:
        ax.set_ylabel('$\\langle\\rho\\rangle$')
        ax.legend(fontsize=7, ncol=2)
    ax.set_xlim(1.0, 2.0)
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0, color='grey', lw=0.5, ls=':')
    ax.axhline(1, color='grey', lw=0.5, ls=':')

fig3.suptitle(
    f'Correlated (—) vs uncorrelated (- -), $\\theta={TH_PLOT}$',
)
fig3.savefig('figures/03-marginals-corr-uncorr.pdf', bbox_inches='tight')
fig3.savefig('figures/03-marginals-corr-uncorr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03-marginals-corr-uncorr.{pdf,png}')

## Observations

*(Fill in after running)*

- Does uncorrelation between layers enhance or reduce cooperation?
- Is the effect of uncorrelation larger for small or large L?
- Does the qualitative effect of L (more circles → more cooperation) survive
  in the uncorrelated case?
- Are there network-topology-dependent differences (BA vs ER, z=4 vs z=16)?